In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [2]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# Step 1: Function to run the experiment
def run_experiment(
    vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name
):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(
            ngram_range=ngram_range, max_features=vectorizer_max_features
        )
    else:
        vectorizer = TfidfVectorizer(
            ngram_range=ngram_range, max_features=vectorizer_max_features
        )

    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_comment"],
        df["category"],
        test_size=0.2,
        random_state=42,
        stratify=df["category"],
    )

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    n_estimators = 200
    max_depth = 15

    # Initialize and train the model
    model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)

    # Step 5: Make predictions and evaluate
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    # Print results summary to console
    run_name = f"{vectorizer_name}_{ngram_range}_RandomForest"
    print("=" * 60)
    print(f"EXPERIMENT: {run_name}")
    print(f"Parameters: max_features={vectorizer_max_features}")
    print("-" * 60)
    print(f"Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("=" * 60 + "\n")

    # Generate and save confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")

    # Saved with a dynamic name to avoid overwriting previous loops
    plot_filename = f"confusion_matrix_{vectorizer_name}_{ngram_range[0]}_{ngram_range[1]}.png"
    plt.savefig(plot_filename)
    plt.close()

    # Optional: return the components if you need to use them later
    return model, accuracy


# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000  # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")


EXPERIMENT: BoW_(1, 1)_RandomForest
Parameters: max_features=5000
------------------------------------------------------------
Accuracy: 0.6480

Classification Report:
              precision    recall  f1-score   support

          -1       0.97      0.02      0.04      1650
           0       0.66      0.84      0.74      2529
           1       0.64      0.83      0.72      3154

    accuracy                           0.65      7333
   macro avg       0.76      0.56      0.50      7333
weighted avg       0.72      0.65      0.57      7333


EXPERIMENT: TF-IDF_(1, 1)_RandomForest
Parameters: max_features=5000
------------------------------------------------------------
Accuracy: 0.6491

Classification Report:
              precision    recall  f1-score   support

          -1       0.97      0.02      0.04      1650
           0       0.66      0.84      0.74      2529
           1       0.64      0.83      0.72      3154

    accuracy                           0.65      7333
   macr